# VC2 · Demo en directo — Smart City
## "El panel de tráfico tarda 40 minutos en actualizarse. Vamos a arreglarlo."

**Esto NO es la actividad evaluable.** Es la demostración de la videoconferencia del NF2.
Tu actividad va de una plataforma de *gaming* y usa otros datos, otras columnas y otros
umbrales. Aquí puedes ejecutar, romper y experimentar: **no se entrega**.

**Antes de ejecutar nada**, una sola vez, desde la raíz del repositorio:

```bash
python demo/nf2_smartcity/preparar_demo.py
```

---

### El encargo

El panel de control de tráfico de la ciudad se alimenta de **2 millones de mediciones** de
sensores. Cada vez que un técnico lo abre, tarda **40 minutos** en recalcular. Es inservible
para tomar decisiones: cuando el número aparece, el atasco ya se ha disuelto.

Nos han pedido dos cosas:
1. Que **vaya rápido**.
2. Que responda a una pregunta que el panel actual no sabe contestar: *"¿hay alguna vía que se
   sature **a horas concretas**, aunque su media diaria parezca normal?"*

Vamos a hacerlo con Spark. Y por el camino vamos a ver **por qué** Spark hace lo que hace —y,
al final, **si de verdad hacía falta**.

In [ ]:
import os, time, json
from pathlib import Path

# Rutas ancladas a ESTE cuaderno (funciona desde su carpeta o desde la raíz del repo)
try:
    BASE = Path(__file__).parent
except NameError:
    BASE = Path.cwd()
    if BASE.name != "nf2_smartcity":
        BASE = BASE / "demo" / "nf2_smartcity"
RAW = BASE / "raw"

assert (RAW / "mediciones.parquet").exists(), (
    "No encuentro los datos. Ejecuta una vez, desde la raíz del repo:\n"
    "    python demo/nf2_smartcity/preparar_demo.py"
)

def cron(fn):
    t = time.perf_counter(); r = fn(); return round(time.perf_counter() - t, 2), r

print("Datos listos en:", BASE)

---
# ACTO 1 · Arranca el motor (y mira lo que NO pasa)

Lo primero es levantar Spark. Fíjate en el reloj: esto tarda. Guárdate esa sensación para el
final de la sesión, porque va a ser el remate.

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

t0 = time.time()
spark = (SparkSession.builder
         .master("local[*]")              # <- toda la magia y toda la trampa están aquí
         .appName("panel-trafico")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Arrancar la sesión de Spark: {time.time()-t0:.1f} s")
print("Spark UI:", spark.sparkContext.uiWebUrl)   # <- abre este enlace en otra pestaña

> **Abre la Spark UI** (el enlace de arriba; en Codespaces, pestaña *Ports* → puerto 4040).
> Déjala abierta en otra ventana: es donde vamos a *ver* lo que Spark hace por dentro.

Sobre ese `local[*]`: significa **"ejecuta Spark en esta máquina, usando todos sus núcleos"**.
No hay clúster, no hay 50 nodos, no hay red. Hay un Codespace con un puñado de cores
**simulando** ser un clúster. Lo importante: **el código es idéntico** al que correría en 500
nodos. Volveremos a esto, porque cambia la interpretación de todo lo que midamos hoy.

In [ ]:
mediciones = spark.read.parquet(str(RAW / "mediciones.parquet"))
vias = spark.read.option("header", True).option("inferSchema", True).csv(str(RAW / "vias.csv"))

print("mediciones:", mediciones.count(), "filas")
mediciones.show(5)
vias.show(5)

---
# ACTO 2 · Pereza: la orden que no se ejecuta

Aquí está la idea que más cuesta a quien viene de pandas. En pandas, cada línea se ejecuta al
escribirla. En Spark, **no**.

Voy a pedir un filtro sobre 2 millones de filas. Cronometra.

In [ ]:
t, criticos = cron(lambda: mediciones.filter(F.col("valor") > 90).select("id_via", "hora", "valor"))
print(f"Definir el filtro sobre 2.000.000 de filas: {t} s")

**Cero coma cero segundos.** ¿Ha filtrado dos millones de filas en ese tiempo? No. **No ha
filtrado nada.**

Lo que ha hecho es **apuntar la orden en una receta**. A eso se le llama **transformación**:
describe un cambio y devuelve otro DataFrame, pero **no ejecuta nada**. La receta se cocina solo
cuando pides un resultado de verdad —una **acción**—:

In [ ]:
t, n = cron(lambda: criticos.count())     # count() SÍ es una acción: dispara todo
print(f"Ahora sí, contar los críticos: {t} s  ->  {n:,} mediciones por encima del umbral")

Esa es la pareja que se examina: **transformación** (perezosa, no ejecuta: `filter`, `select`,
`groupBy`, `join`) y **acción** (dispara la ejecución: `count`, `collect`, `show`, `write`).

**¿Y para qué sirve ser perezoso?** Para esto: como Spark ve **la receta entera** antes de
cocinar, su optimizador (**Catalyst**) puede reordenarla. Si escribes el filtro al final,
Catalyst lo **empuja al principio** para mover menos datos. En pandas eso es imposible: cuando
quiere optimizar, ya has ejecutado. **La pereza es la condición que hace posible la
optimización.** Esa frase, tal cual, es la que puntúa en el examen.

---
# ACTO 3 · El linaje: cómo sobrevive Spark a que se caiga una máquina

Pregunta de examen: en mitad de un job de 4 horas se cae un nodo con tres particiones. ¿Qué
hace Spark? La respuesta ingenua es "tenía una copia". **No.** Spark no copia los datos —sería
carísimo—. Guarda algo mucho más barato: **la receta para recalcularlos**. Eso es el **linaje**.

Vamos a verlo.

In [ ]:
# Construimos un DataFrame derivado con varios pasos: join + filtro + columna nueva
derivado = (mediciones
            .join(vias, "id_via")
            .filter(F.col("tipo") == "trafico")
            .withColumn("es_pico", F.col("hora").between(7, 9)))

print("El linaje: la receta completa para reconstruir CUALQUIER partición\n")
print(derivado.rdd.toDebugString().decode()[:900])

Eso que ves es **la genealogía de cada partición**: "para reconstruir este trozo, lee el
Parquet, haz el join, filtra, añade la columna". Si una máquina se cae, Spark **no restaura una
copia**: mira el linaje y **vuelve a fabricar solo las particiones perdidas**, desde el último
punto disponible. Las demás máquinas ni se enteran.

Y esto solo es posible porque **los DataFrames son inmutables**: como nadie ha modificado los
datos por debajo, repetir la receta da exactamente el mismo resultado. **Inmutabilidad → linaje
→ tolerancia a fallos sin replicar.** Ese es el encadenamiento que pide el RA2.

---
# ACTO 4 · La operación cara: contar los *shuffles*, no las líneas

Ahora la pregunta del panel: *NO₂... perdón, tráfico medio por distrito*. Suena inocente. No lo
es. Fíjate en la Spark UI mientras corre.

In [ ]:
t, por_distrito = cron(lambda:
    mediciones.filter(F.col("tipo") == "trafico")
              .join(vias, "id_via")
              .groupBy("distrito").agg(F.avg("valor").alias("media"))
              .orderBy(F.desc("media"))
              .collect())
print(f"tráfico medio por distrito: {t} s")
for r in por_distrito:
    print(f"  {r['distrito']:<10} {r['media']:.1f}")

Mira el DAG en la Spark UI (pestaña *SQL / DataFrame* → tu última consulta). Vas a ver **varias
cajas** encadenadas. **Cada caja es un *stage*. Cada frontera entre cajas es un *shuffle*.**

Un **shuffle** es reorganizar los datos por la red para juntar las filas que van juntas: todos
los `Centro` en una máquina, todos los `Ensanche` en otra. Y es **la operación cara** de Spark
—escribe a disco, manda por red, vuelve a leer, para *todos* los datos—. Aquí hay dos: el
`join` y el `groupBy` (el `orderBy` mete otro más).

> **La regla que se te tiene que quedar:** en Spark, la unidad de coste **no es el número de
> operaciones, es el número de shuffles**. Un pipeline con 30 filtros y 1 shuffle es más rápido
> que uno con 3 filtros y 2 shuffles. Contar stages en la UI es contar tu factura.

---
# ACTO 5 · La pregunta que una media global no sabe responder

Volvemos al encargo. *"¿Hay alguna vía que se sature a horas concretas, aunque su media diaria
parezca normal?"* Probemos primero por las bravas: media de tráfico por vía.

In [ ]:
media_via = (mediciones.filter(F.col("tipo") == "trafico")
             .groupBy("id_via").agg(F.avg("valor").alias("media"))
             .orderBy(F.desc("media")))
media_via.show(5)

La vía 7 asoma la cabeza (~64 frente a ~60), pero es **casi indistinguible** del resto. Si
ordenas por esa columna, no salta ninguna alarma. **La media diaria esconde el problema**,
porque diluye una hora mala entre 23 horas normales.

La pregunta de verdad no es *"¿cuánto tráfico tiene la vía 7?"*. Es *"¿es raro este valor
**para esta vía a esta hora**?"*. Y eso **no se responde con una media global**. Se responde
con una **función de ventana**.

In [ ]:
# Enfoque 1 (el que sale solo): media por (via, hora) y luego join de vuelta. DOS shuffles.
t1, _ = cron(lambda:
    mediciones.join(
        mediciones.groupBy("id_via", "hora").agg(F.avg("valor").alias("media")),
        ["id_via", "hora"]
    ).write.mode("overwrite").format("noop").save())

# Enfoque 2: función de ventana. La media se calcula y se adjunta en la MISMA pasada. UN shuffle.
w = Window.partitionBy("id_via", "hora")
t2, _ = cron(lambda:
    mediciones.withColumn("media_vh", F.avg("valor").over(w))
              .write.mode("overwrite").format("noop").save())

print(f"groupBy + join (2 shuffles): {t1} s")
print(f"función de ventana (1 shuffle): {t2} s")
print(f"-> {round(t1/t2, 1)}x  (en local; en un clúster real la diferencia es mayor, ver nota)")

> ⚠️ **Nota honesta sobre ese número** (y es en sí misma una lección del núcleo). En un clúster
> de verdad, esta mejora suele ser **de más del doble**. Aquí sale menos espectacular porque
> estamos en `local[*]`: el shuffle mueve datos **entre hilos de esta máquina, no por la red**,
> así que **lo más caro de Spark aquí es barato**. La lección se mantiene intacta —un shuffle
> menos es siempre mejor— pero el *tamaño* del efecto solo se ve cuando la red entra en juego.

Y ahora usamos la ventana para responder de verdad: marcamos cada medición con **cuánto se
desvía de lo normal para su vía y su hora**.

In [ ]:
anomalias = (mediciones.filter(F.col("tipo") == "trafico")
             .withColumn("media_vh", F.avg("valor").over(w))
             .withColumn("desvio", F.col("valor") / F.col("media_vh"))
             .groupBy("id_via", "hora")
             .agg(F.round(F.avg("valor"), 1).alias("valor_medio"))
             .orderBy(F.desc("valor_medio")))
anomalias.show(5)

**Ahí está.** `id_via = 7`, `hora = 8`: **más del doble** de su propio comportamiento normal.
Un atasco sistemático en hora punta que la media diaria **borraba por completo**. Esto es un
*insight* accionable: no "la vía 7 tiene tráfico", sino "**la vía 7 se colapsa a las 8 h**" —que
es lo que permite poner un semáforo inteligente o desviar una línea de bus.

Esto, además, no es solo una optimización: una media global **no podía** dar esta respuesta. La
función de ventana no es un truco de velocidad, es **la herramienta lógica correcta** para las
preguntas contextuales.

---
# ACTO 6 · El mismo código, pero para datos que aún no han llegado

Todo lo anterior es *batch*: proceso un histórico que ya está. Pero el panel es en vivo. Los
sensores **siguen emitiendo**. ¿Reescribo todo para streaming?

**No.** Y esta es una de las mejores ideas de Spark: *Structured Streaming* trata un flujo como
**una tabla a la que no paran de llegar filas**. El código es casi el mismo que el batch.

In [ ]:
esquema = "id_medicion long, id_via int, hora int, timestamp string, tipo string, valor double"

flujo = (spark.readStream.schema(esquema)          # <- en streaming el esquema se DECLARA
         .json(str(RAW / "stream_in")))            #    (no se puede inferir de lo que no ha llegado)

conteo = flujo.groupBy("tipo").count()             # <- exactamente igual que en batch

q = (conteo.writeStream.format("memory").queryName("panel_vivo")
     .outputMode("complete").trigger(availableNow=True).start())
q.awaitTermination()

spark.sql("SELECT tipo, count FROM panel_vivo ORDER BY count DESC").show()

Fíjate en lo único que ha cambiado respecto al batch: `readStream` en vez de `read`, el
esquema **declarado** (no se puede inferir de datos que aún no existen), y un `writeStream` con
su *sink*. El `groupBy("tipo").count()` del medio es **idéntico**. Ese es el regalo: aprendes el
modelo una vez y sirve para las dos velocidades del negocio.

*(Nota: `trigger(availableNow=True)` procesa lo que haya y para, ideal para demo. En producción
el trigger es continuo.)*

---
# ACTO 7 · La pregunta incómoda: ¿de verdad hacía falta Spark?

Hemos montado particiones, shuffles, DAG, linaje, streaming. Impresionante. Y ahora la pregunta
que un buen ingeniero se hace **antes** de todo esto, no después.

Nuestra consulta estrella era *tráfico medio por tipo*. La hemos hecho en Spark. Vamos a
hacerla en **DuckDB** —un motor de un solo nodo, sin clúster, sin arranque—, sobre exactamente
el mismo fichero, y a cronometrar las dos.

In [ ]:
import duckdb

t_duck, r_duck = cron(lambda: duckdb.sql(
    f"SELECT tipo, round(avg(valor),2) AS media "
    f"FROM '{RAW / 'mediciones.parquet'}' GROUP BY tipo ORDER BY tipo"
).fetchall())

t_spark, r_spark = cron(lambda:
    mediciones.groupBy("tipo").agg(F.round(F.avg("valor"), 2).alias("media"))
              .orderBy("tipo").collect())

print(f"DuckDB : {t_duck:>6.3f} s   {r_duck}")
print(f"Spark  : {t_spark:>6.3f} s   (+ los ~15 s que tardó en arrancar la sesión al principio)")
print()
print("Mismo resultado. Uno necesitó un 'clúster' y 15 s de arranque. El otro, una línea de SQL.")

Léelo despacio, porque es la tesis del núcleo:

> **DuckDB y Polars no vienen a sustituir a Spark. Vienen a sustituir a pandas.**

Nuestros 2 millones de filas **caben de sobra en una máquina**. Para eso, montar Spark es
llevar una grúa para colgar un cuadro: el arranque, la configuración, el modelo mental del
shuffle... todo ese coste, para un dato que DuckDB resuelve en milisegundos.

**La regla que te llevas del NF2:**

> **Si cabe en una máquina, no montes un clúster.**
> Y si NO cabe —terabytes—, o necesitas *streaming con estado*, **Spark sigue sin rival**.

Entonces, ¿por qué has aprendido Spark? Porque **Spark es donde los conceptos se ven**.
Particiones, DAG, linaje, shuffle: son las ideas del cómputo distribuido, y no se aprenden en un
motor que las esconde. Un ingeniero que sabe Spark pero no sabe **cuándo no usarlo** es peor
ingeniero que uno que sabe las dos cosas.

---

## Lo que ha pasado en esta hora

Un panel lento. Y por el camino, las seis propiedades del cómputo distribuido:

| Acto | La idea | Teoría | Se examina |
|---|---|---|---|
| 2 | Pereza: transformación vs acción | §2.1 | ✅ bloque 1 |
| 3 | Linaje: tolerancia sin copiar | §2.2 | ✅ bloque 4 |
| 4 | Shuffle: la unidad de coste | §2.3 | (terminología) |
| 5 | Función de ventana: 1 shuffle, no 2 | §2.3 | — |
| 6 | Batch y streaming, mismo código | §2.5 | ✅ bloque 3 |
| 7 | ¿Necesitas Spark? DuckDB | §2.6 | ✅ autocomp. 4 |

**Y ahora te toca a ti**, con los eventos de un juego. Mismo trabajo, otros datos. La actividad
te pedirá abrir la Spark UI y **contar los stages tú mismo**: ya sabes qué estás mirando.